# 🔀 Matriz de Cruce: Clusters k=8 × Segmentos de Propensión

**Objetivo**: Cruzar los 8 clusters con los segmentos de propensión de:
- Pension Plan
- Investment

**Output**: Matrices mostrando cómo se distribuyen los clientes de cada cluster en los segmentos de propensión (valores absolutos + porcentajes)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Cargar Datos

In [ ]:
# Cargar clusters k=8
df_clusters = pd.read_csv('../data/customer_clusters_k8.csv')

# Cargar propensión pension plan
df_pension = pd.read_csv('../../classification_model/data/processed/df_pension_plan_scored.csv')

# Cargar propensión investment
df_investment = pd.read_csv('../../classification_model/data/processed/df_inversion_family_scored.csv')

print(f"Clusters k=8: {len(df_clusters):,} clientes")
print(f"Propensión pension: {len(df_pension):,} clientes")
print(f"Propensión investment: {len(df_investment):,} clientes")

---

# 📊 PARTE 1: PENSION PLAN

---

## 2. Merge Pension Plan

In [ ]:
# Merge por pk_cid
df_pension_merged = df_clusters.merge(
    df_pension[['pk_cid', 'propensity_score', 'propensity_segment']], 
    on='pk_cid', 
    how='inner'
)

print(f"Datos combinados PENSION: {len(df_pension_merged):,} clientes")
print(f"\nDistribución de Segmentos de Propensión PENSION:\n")
print(df_pension_merged['propensity_segment'].value_counts().sort_index())
print("\nPorcentajes:")
print((df_pension_merged['propensity_segment'].value_counts(normalize=True).sort_index() * 100).round(2))

## 3. PENSION - Matriz Valores Absolutos

In [ ]:
# Crear tabla de contingencia - VALORES ABSOLUTOS
pension_abs = pd.crosstab(
    index=df_pension_merged['cluster_name'],
    columns=df_pension_merged['propensity_segment'],
    margins=True,
    margins_name='TOTAL'
)

print("\n" + "="*80)
print("📊 PENSION PLAN - MATRIZ VALORES ABSOLUTOS (N° Clientes)")
print("="*80)
print(pension_abs)

# Guardar
pension_abs.to_csv('../data/pension_cluster_absolutos_k8.csv')
print("\n✅ Guardado: pension_cluster_absolutos_k8.csv")

## 4. PENSION - Matriz Porcentajes por Fila

In [ ]:
# Porcentajes por fila (% de cada cluster que cae en cada segmento)
pension_pct_fila = pd.crosstab(
    index=df_pension_merged['cluster_name'],
    columns=df_pension_merged['propensity_segment'],
    normalize='index'
) * 100

print("\n" + "="*80)
print("📊 PENSION PLAN - MATRIZ % POR CLUSTER (Fila)")
print("="*80)
print(pension_pct_fila.round(1))

# Guardar
pension_pct_fila.to_csv('../data/pension_cluster_pct_fila_k8.csv')
print("\n✅ Guardado: pension_cluster_pct_fila_k8.csv")

## 5. PENSION - Matriz Porcentajes por Columna

In [ ]:
# Porcentajes por columna (% de cada segmento que pertenece a cada cluster)
pension_pct_col = pd.crosstab(
    index=df_pension_merged['cluster_name'],
    columns=df_pension_merged['propensity_segment'],
    normalize='columns'
) * 100

print("\n" + "="*80)
print("📊 PENSION PLAN - MATRIZ % POR SEGMENTO (Columna)")
print("="*80)
print(pension_pct_col.round(1))

# Guardar
pension_pct_col.to_csv('../data/pension_cluster_pct_col_k8.csv')
print("\n✅ Guardado: pension_cluster_pct_col_k8.csv")

## 6. PENSION - Heatmap

In [ ]:
# Ordenar segmentos de propensión (Top -> Low)
segment_order = ['Top', 'High', 'Medium', 'Mid-Low', 'Low']
existing_segments = [s for s in segment_order if s in pension_pct_fila.columns]

pension_ordered = pension_pct_fila[existing_segments]

# Crear heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    pension_ordered,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    cbar_kws={'label': '% Clientes del Cluster'},
    linewidths=0.5,
    vmin=0,
    vmax=100
)
plt.title('PENSION PLAN: Distribución de Clusters por Segmento de Propensión\n(% de cada cluster en cada segmento)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Segmento de Propensión', fontsize=12)
plt.ylabel('Cluster', fontsize=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../visualizations/heatmap_pension_cluster_k8.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Heatmap guardado: heatmap_pension_cluster_k8.png")

## 7. PENSION - Resumen Ejecutivo

In [ ]:
# Crear resumen para presentación
resumen_pension = []

for cluster_name in sorted(df_pension_merged['cluster_name'].unique()):
    cluster_data = df_pension_merged[df_pension_merged['cluster_name'] == cluster_name]
    
    # Segmento predominante
    segmento_top = cluster_data['propensity_segment'].mode()[0]
    pct_top = (cluster_data['propensity_segment'] == segmento_top).mean() * 100
    
    # Calcular % en alto valor (Top + High)
    alto_valor_pct = cluster_data['propensity_segment'].isin(['Top', 'High']).mean() * 100
    alto_valor_n = cluster_data['propensity_segment'].isin(['Top', 'High']).sum()
    
    resumen_pension.append({
        'Cluster': cluster_name,
        'N_Clientes': len(cluster_data),
        'N_Alto_Valor': alto_valor_n,
        '%_Alto_Valor': alto_valor_pct,
        'Score_Promedio': cluster_data['propensity_score'].mean()
    })

df_resumen_pension = pd.DataFrame(resumen_pension)
df_resumen_pension = df_resumen_pension.sort_values('%_Alto_Valor', ascending=False)

print("\n" + "="*80)
print("📊 PENSION PLAN - RESUMEN EJECUTIVO POR CLUSTER")
print("="*80)
print(df_resumen_pension.to_string(index=False))

# Guardar
df_resumen_pension.to_csv('../data/resumen_pension_cluster_k8.csv', index=False)
print("\n✅ Guardado: resumen_pension_cluster_k8.csv")

---

# 💼 PARTE 2: INVESTMENT

---

## 8. Merge Investment

In [ ]:
# Merge por pk_cid
df_investment_merged = df_clusters.merge(
    df_investment[['pk_cid', 'propensity_score', 'propensity_segment']], 
    on='pk_cid', 
    how='inner'
)

print(f"Datos combinados INVESTMENT: {len(df_investment_merged):,} clientes")
print(f"\nDistribución de Segmentos de Propensión INVESTMENT:\n")
print(df_investment_merged['propensity_segment'].value_counts().sort_index())
print("\nPorcentajes:")
print((df_investment_merged['propensity_segment'].value_counts(normalize=True).sort_index() * 100).round(2))

## 9. INVESTMENT - Matriz Valores Absolutos

In [ ]:
# Crear tabla de contingencia - VALORES ABSOLUTOS
investment_abs = pd.crosstab(
    index=df_investment_merged['cluster_name'],
    columns=df_investment_merged['propensity_segment'],
    margins=True,
    margins_name='TOTAL'
)

print("\n" + "="*80)
print("📊 INVESTMENT - MATRIZ VALORES ABSOLUTOS (N° Clientes)")
print("="*80)
print(investment_abs)

# Guardar
investment_abs.to_csv('../data/investment_cluster_absolutos_k8.csv')
print("\n✅ Guardado: investment_cluster_absolutos_k8.csv")

## 10. INVESTMENT - Matriz Porcentajes por Fila

In [ ]:
# Porcentajes por fila (% de cada cluster que cae en cada segmento)
investment_pct_fila = pd.crosstab(
    index=df_investment_merged['cluster_name'],
    columns=df_investment_merged['propensity_segment'],
    normalize='index'
) * 100

print("\n" + "="*80)
print("📊 INVESTMENT - MATRIZ % POR CLUSTER (Fila)")
print("="*80)
print(investment_pct_fila.round(1))

# Guardar
investment_pct_fila.to_csv('../data/investment_cluster_pct_fila_k8.csv')
print("\n✅ Guardado: investment_cluster_pct_fila_k8.csv")

## 11. INVESTMENT - Matriz Porcentajes por Columna

In [ ]:
# Porcentajes por columna (% de cada segmento que pertenece a cada cluster)
investment_pct_col = pd.crosstab(
    index=df_investment_merged['cluster_name'],
    columns=df_investment_merged['propensity_segment'],
    normalize='columns'
) * 100

print("\n" + "="*80)
print("📊 INVESTMENT - MATRIZ % POR SEGMENTO (Columna)")
print("="*80)
print(investment_pct_col.round(1))

# Guardar
investment_pct_col.to_csv('../data/investment_cluster_pct_col_k8.csv')
print("\n✅ Guardado: investment_cluster_pct_col_k8.csv")

## 12. INVESTMENT - Heatmap

In [ ]:
# Ordenar segmentos de propensión (Top -> Low)
segment_order = ['Top', 'High', 'Medium', 'Mid-Low', 'Low']
existing_segments_inv = [s for s in segment_order if s in investment_pct_fila.columns]

investment_ordered = investment_pct_fila[existing_segments_inv]

# Crear heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    investment_ordered,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    cbar_kws={'label': '% Clientes del Cluster'},
    linewidths=0.5,
    vmin=0,
    vmax=100
)
plt.title('INVESTMENT: Distribución de Clusters por Segmento de Propensión\n(% de cada cluster en cada segmento)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Segmento de Propensión', fontsize=12)
plt.ylabel('Cluster', fontsize=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../visualizations/heatmap_investment_cluster_k8.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Heatmap guardado: heatmap_investment_cluster_k8.png")

## 13. INVESTMENT - Resumen Ejecutivo

In [ ]:
# Crear resumen para presentación
resumen_investment = []

for cluster_name in sorted(df_investment_merged['cluster_name'].unique()):
    cluster_data = df_investment_merged[df_investment_merged['cluster_name'] == cluster_name]
    
    # Segmento predominante
    segmento_top = cluster_data['propensity_segment'].mode()[0]
    pct_top = (cluster_data['propensity_segment'] == segmento_top).mean() * 100
    
    # Calcular % en alto valor (Top + High)
    alto_valor_pct = cluster_data['propensity_segment'].isin(['Top', 'High']).mean() * 100
    alto_valor_n = cluster_data['propensity_segment'].isin(['Top', 'High']).sum()
    
    resumen_investment.append({
        'Cluster': cluster_name,
        'N_Clientes': len(cluster_data),
        'N_Alto_Valor': alto_valor_n,
        '%_Alto_Valor': alto_valor_pct,
        'Score_Promedio': cluster_data['propensity_score'].mean()
    })

df_resumen_investment = pd.DataFrame(resumen_investment)
df_resumen_investment = df_resumen_investment.sort_values('%_Alto_Valor', ascending=False)

print("\n" + "="*80)
print("📊 INVESTMENT - RESUMEN EJECUTIVO POR CLUSTER")
print("="*80)
print(df_resumen_investment.to_string(index=False))

# Guardar
df_resumen_investment.to_csv('../data/resumen_investment_cluster_k8.csv', index=False)
print("\n✅ Guardado: resumen_investment_cluster_k8.csv")

---

# 🔄 PARTE 3: COMPARATIVA PENSION vs INVESTMENT

---

## 14. Comparativa de Alto Valor por Cluster

In [ ]:
# Combinar resúmenes
comparativa = df_resumen_pension[['Cluster', 'N_Alto_Valor', '%_Alto_Valor', 'Score_Promedio']].copy()
comparativa.columns = ['Cluster', 'N_Alto_Valor_Pension', '%_Alto_Valor_Pension', 'Score_Pension']

inv_cols = df_resumen_investment[['Cluster', 'N_Alto_Valor', '%_Alto_Valor', 'Score_Promedio']].copy()
inv_cols.columns = ['Cluster', 'N_Alto_Valor_Investment', '%_Alto_Valor_Investment', 'Score_Investment']

comparativa = comparativa.merge(inv_cols, on='Cluster')

# Ordenar por promedio de alto valor
comparativa['%_Promedio'] = (comparativa['%_Alto_Valor_Pension'] + comparativa['%_Alto_Valor_Investment']) / 2
comparativa = comparativa.sort_values('%_Promedio', ascending=False)

print("\n" + "="*100)
print("📊 COMPARATIVA PENSION vs INVESTMENT - ALTO VALOR (Top + High)")
print("="*100)
print(comparativa.to_string(index=False))

# Guardar
comparativa.to_csv('../data/comparativa_pension_investment_k8.csv', index=False)
print("\n✅ Guardado: comparativa_pension_investment_k8.csv")

## 15. Top Insights Comparativa

In [ ]:
print("\n" + "="*80)
print("🎯 TOP INSIGHTS - COMPARATIVA")
print("="*80)

# 1. Mejor cluster para pension
best_pension = comparativa.loc[comparativa['%_Alto_Valor_Pension'].idxmax()]
print(f"\n1. Mejor cluster para PENSION PLAN:")
print(f"   → {best_pension['Cluster']}")
print(f"   → {best_pension['%_Alto_Valor_Pension']:.1f}% en alto valor ({int(best_pension['N_Alto_Valor_Pension']):,} clientes)")

# 2. Mejor cluster para investment
best_investment = comparativa.loc[comparativa['%_Alto_Valor_Investment'].idxmax()]
print(f"\n2. Mejor cluster para INVESTMENT:")
print(f"   → {best_investment['Cluster']}")
print(f"   → {best_investment['%_Alto_Valor_Investment']:.1f}% en alto valor ({int(best_investment['N_Alto_Valor_Investment']):,} clientes)")

# 3. Cluster más balanceado
comparativa['Diferencia'] = abs(comparativa['%_Alto_Valor_Pension'] - comparativa['%_Alto_Valor_Investment'])
most_balanced = comparativa.loc[comparativa['Diferencia'].idxmin()]
print(f"\n3. Cluster más balanceado (similar propensión en ambos):")
print(f"   → {most_balanced['Cluster']}")
print(f"   → Pension: {most_balanced['%_Alto_Valor_Pension']:.1f}%, Investment: {most_balanced['%_Alto_Valor_Investment']:.1f}%")
print(f"   → Diferencia: {most_balanced['Diferencia']:.1f}pp")

print("\n" + "="*80)

---

## ✅ ANÁLISIS COMPLETADO

### Archivos generados PENSION:
1. `pension_cluster_absolutos_k8.csv` - Valores absolutos N° clientes
2. `pension_cluster_pct_fila_k8.csv` - % por cluster
3. `pension_cluster_pct_col_k8.csv` - % por segmento
4. `resumen_pension_cluster_k8.csv` - Resumen ejecutivo
5. `heatmap_pension_cluster_k8.png` - Visualización

### Archivos generados INVESTMENT:
1. `investment_cluster_absolutos_k8.csv` - Valores absolutos N° clientes
2. `investment_cluster_pct_fila_k8.csv` - % por cluster
3. `investment_cluster_pct_col_k8.csv` - % por segmento
4. `resumen_investment_cluster_k8.csv` - Resumen ejecutivo
5. `heatmap_investment_cluster_k8.png` - Visualización

### Archivo COMPARATIVA:
- `comparativa_pension_investment_k8.csv` - Comparación lado a lado

### Interpretación:
- **Matriz valores absolutos**: Número exacto de clientes en cada cluster×segmento
- **Matriz % por fila**: "Del cluster X, el Y% está en el segmento Top"
- **Matriz % por columna**: "Del segmento Top, el Y% pertenece al cluster X"
- **Resumen**: N° clientes y % en alto valor (Top+High) por cluster